# CFRP Anomaly Detection — Baselines

Three unsupervised PyOD baselines on the CFRP dataset:

- **Isolation Forest** — random partitioning; cheap, strong default.
- **LOF** (Local Outlier Factor) — density-based.
- **ECOD** — empirical CDF, parameter-free.

All three score every row with a continuous anomaly score. We pick the decision threshold by maximizing F1 on the **validation** split, then evaluate on the held-out **test** split. Threshold-independent metrics (ROC-AUC, PR-AUC) are reported in parallel.

Every run is logged to MLflow (`./mlruns`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import mlflow

from src.data_loader import load_cfrp, split_70_20_10
from src.preprocessing import prepare
from src.modeling import build_detectors, run_baseline
from src.mlflow_utils import load_config, init_mlflow, log_split_sizes

config = load_config(PROJECT_ROOT / "config.yaml")
init_mlflow(config)
config

C:\Users\feris\Desktop\Julia_ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\feris\Desktop\Julia_ML\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'project': {'name': 'smartmanu-ad-cfrp', 'random_seed': 42},
 'dataset': {'name': 'CFRP',
  'source_url': 'https://github.com/SmartManuAD/Smart-Manufacturing-AD',
  'raw_path': 'data/raw/5_cfrp.npz',
  'processed_dir': 'data/processed'},
 'split': {'train': 0.7, 'val': 0.2, 'test': 0.1, 'stratify_on_label': True},
 'mlflow': {'tracking_uri': 'file:./mlruns',
  'experiment_name': 'cfrp-anomaly-detection'},
 '_project_root': 'C:\\Users\\feris\\Desktop\\Julia_ML'}

## 1. Load and split

In [2]:
raw_path = PROJECT_ROOT / config["dataset"]["raw_path"]
X, y = load_cfrp(raw_path)
print(f"X={X.shape}  y={y.shape}  anomaly rate={y.mean():.4%}")

X_train, y_train, X_val, y_val, X_test, y_test = split_70_20_10(
    X, y,
    seed=config["project"]["random_seed"],
    stratify=config["split"]["stratify_on_label"],
)

split_table = pd.DataFrame(
    {
        "n": [len(y_train), len(y_val), len(y_test)],
        "share": [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
        "pos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
        "pos_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "val", "test"],
)
split_table

X=(52268, 49)  y=(52268,)  anomaly rate=0.5013%


,n,share,pos,pos_rate
train,36587,0.699989,183,0.005002
val,10454,0.200008,53,0.005070
test,5227,0.100004,26,0.004974


## 2. Preprocess (impute then scale, fit on train only)

In [3]:
data = prepare(X_train, y_train, X_val, y_val, X_test, y_test,
               impute="median", scaler="robust")
print("Shapes after preprocessing:")
print(f"  train: {data.X_train.shape}")
print(f"  val:   {data.X_val.shape}")
print(f"  test:  {data.X_test.shape}")

Shapes after preprocessing:
  train: (36587, 49)
  val:   (10454, 49)
  test:  (5227, 49)


## 3. Log split sizes once as a parent run

In [4]:
with mlflow.start_run(run_name="data_setup"):
    mlflow.set_tag("stage", "data")
    log_split_sizes(data.X_train, data.X_val, data.X_test,
                    data.y_train, data.y_val, data.y_test)
    mlflow.log_param("impute_strategy", "median")
    mlflow.log_param("scaler", "robust")
    mlflow.log_param("random_seed", config["project"]["random_seed"])
    mlflow.log_artifact(str(PROJECT_ROOT / "reports" / "dataset_card.md"),
                        artifact_path="dataset")

## 4. Train and evaluate three baselines

Each call below is a separate MLflow run (parameters, metrics, confusion matrix, ROC curve, fitted model, dataset card).

In [5]:
contamination = float(data.y_train.mean())
detectors = build_detectors(contamination=contamination,
                            seed=config["project"]["random_seed"])

results = {}
for name, det in detectors.items():
    print(f"\n=== {name} ===")
    extra_params = {}
    if name == "autoencoder":
        extra_params = {
            "hidden_neuron_list": str(det.hidden_neuron_list),
            "epoch_num": det.epoch_num,
            "batch_size": det.batch_size,
            "dropout_rate": det.dropout_rate,
            "lr": det.lr,
        }
    res = run_baseline(name, det, data, params={"contamination": contamination, **extra_params})
    results[name] = res
    print(f"  val  ROC-AUC={res.val.roc_auc:.4f}  PR-AUC={res.val.pr_auc:.4f}  F1={res.val.f1:.4f}")
    print(f"  test ROC-AUC={res.test.roc_auc:.4f}  PR-AUC={res.test.pr_auc:.4f}  F1={res.test.f1:.4f}")


=== iforest ===


  val  ROC-AUC=0.9085  PR-AUC=0.3743  F1=0.4222
  test ROC-AUC=0.9152  PR-AUC=0.4481  F1=0.4390

=== lof ===


  val  ROC-AUC=0.7170  PR-AUC=0.0317  F1=0.1134
  test ROC-AUC=0.8537  PR-AUC=0.0514  F1=0.1410

=== ecod ===


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.


[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    2.9s remaining:   20.8s
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    3.0s finished


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    0.0s remaining:    0.0s


[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    0.0s finished


  val  ROC-AUC=0.8204  PR-AUC=0.1526  F1=0.2391
  test ROC-AUC=0.8462  PR-AUC=0.2862  F1=0.3137

=== autoencoder ===


  val  ROC-AUC=0.9947  PR-AUC=0.7539  F1=0.6852
  test ROC-AUC=0.9896  PR-AUC=0.6844  F1=0.6415


## 5. Comparison table

In [6]:
rows = []
for name, res in results.items():
    rows.append({
        "model": name,
        "val_ROC-AUC": res.val.roc_auc,
        "val_PR-AUC": res.val.pr_auc,
        "val_F1": res.val.f1,
        "test_ROC-AUC": res.test.roc_auc,
        "test_PR-AUC": res.test.pr_auc,
        "test_precision": res.test.precision,
        "test_recall": res.test.recall,
        "test_F1": res.test.f1,
        "threshold": res.test.threshold,
        "tp": res.test.tp, "fp": res.test.fp,
        "fn": res.test.fn, "tn": res.test.tn,
    })
summary = pd.DataFrame(rows).set_index("model")
summary.style.format("{:.4f}", subset=[c for c in summary.columns if c not in ("tp", "fp", "fn", "tn")])

,val_ROC-AUC,val_PR-AUC,val_F1,test_ROC-AUC,test_PR-AUC,test_precision,test_recall,test_F1,threshold,tp,fp,fn,tn
model,,,,,,,,,,,,,
iforest,0.9085,0.3743,0.4222,0.9152,0.4481,0.6000,0.3462,0.4390,0.0065,9,6,17,5195
lof,0.7170,0.0317,0.1134,0.8537,0.0514,0.0796,0.6154,0.1410,2.7159,16,185,10,5016
ecod,0.8204,0.1526,0.2391,0.8462,0.2862,0.3200,0.3077,0.3137,117.7172,8,17,18,5184
autoencoder,0.9947,0.7539,0.6852,0.9896,0.6844,0.6296,0.6538,0.6415,4.6728,17,10,9,5191


## 7. Hyperparameter sweep — Isolation Forest

Twelve combinations of `n_estimators × max_samples × max_features`. Each combo is logged as its own MLflow run with tag `sweep=iforest_v1` so it can be filtered together with the other sweep runs in the UI.

- TRAIN/VAL/TEST split, scaler, and seed are unchanged — only the model varies.
- The F1-optimal threshold is re-tuned on VAL inside each run (no test leakage).
- Winner is picked by **val F1**, then we *report* its test metrics. Picking by test F1 would be cheating.

In [7]:
from itertools import product
from pyod.models.iforest import IForest

grid = {
    "n_estimators": [100, 200, 400],
    "max_samples":  [256, 1024],
    "max_features": [0.5, 1.0],
}

seed = config["project"]["random_seed"]
sweep_rows = []

for n_est, n_samp, n_feat in product(grid["n_estimators"], grid["max_samples"], grid["max_features"]):
    run_name = f"iforest_n{n_est}_s{n_samp}_f{n_feat}"
    print(f"=== {run_name} ===")
    det = IForest(
        n_estimators=n_est,
        max_samples=n_samp,
        max_features=n_feat,
        contamination=contamination,
        random_state=seed,
        n_jobs=-1,
    )
    res = run_baseline(
        run_name,
        det,
        data,
        params={
            "contamination": contamination,
            "n_estimators": n_est,
            "max_samples": n_samp,
            "max_features": n_feat,
        },
        tags={"sweep": "iforest_v1"},
        model_family="iforest",
    )
    sweep_rows.append({
        "run": run_name,
        "n_estimators": n_est,
        "max_samples": n_samp,
        "max_features": n_feat,
        "val_F1": res.val.f1,
        "test_F1": res.test.f1,
        "test_ROC-AUC": res.test.roc_auc,
        "test_PR-AUC": res.test.pr_auc,
        "test_precision": res.test.precision,
        "test_recall": res.test.recall,
    })
    print(f"  val F1={res.val.f1:.4f}  test F1={res.test.f1:.4f}  test PR-AUC={res.test.pr_auc:.4f}")

sweep_df = pd.DataFrame(sweep_rows).sort_values("val_F1", ascending=False).reset_index(drop=True)
print("\nTop 3 by validation F1:")
print(sweep_df.head(3).to_string(index=False))
sweep_df.style.format("{:.4f}", subset=[c for c in sweep_df.columns if c not in ("run", "n_estimators", "max_samples")])

=== iforest_n100_s256_f0.5 ===


  val F1=0.4174  test F1=0.4918  test PR-AUC=0.4237
=== iforest_n100_s256_f1.0 ===


  val F1=0.4390  test F1=0.3243  test PR-AUC=0.3782
=== iforest_n100_s1024_f0.5 ===


  val F1=0.4359  test F1=0.5000  test PR-AUC=0.5174
=== iforest_n100_s1024_f1.0 ===


  val F1=0.4375  test F1=0.4255  test PR-AUC=0.4336
=== iforest_n200_s256_f0.5 ===


...

  val F1=0.4356  test F1=0.4167  test PR-AUC=0.3970
=== iforest_n400_s256_f1.0 ===


  val F1=0.4706  test F1=0.4103  test PR-AUC=0.4471
=== iforest_n400_s1024_f0.5 ===


  val F1=0.4593  test F1=0.4800  test PR-AUC=0.5490
=== iforest_n400_s1024_f1.0 ===


  val F1=0.4348  test F1=0.5455  test PR-AUC=0.4867

Top 3 by validation F1:
                    run  n_estimators  max_samples  max_features   val_F1  test_F1  test_ROC-AUC  test_PR-AUC  test_precision  test_recall
 iforest_n400_s256_f1.0           400          256           1.0 0.470588 0.410256      0.912325     0.447086        0.615385     0.307692
iforest_n400_s1024_f0.5           400         1024           0.5 0.459259 0.480000      0.937423     0.548974        0.367347     0.692308
iforest_n200_s1024_f0.5           200         1024           0.5 0.447059 0.550000      0.933755     0.527789        0.785714     0.423077


,run,n_estimators,max_samples,max_features,val_F1,test_F1,test_ROC-AUC,test_PR-AUC,test_precision,test_recall
0,iforest_n400_s256_f1.0,400,256,1.0000,0.4706,0.4103,0.9123,0.4471,0.6154,0.3077
1,iforest_n400_s1024_f0.5,400,1024,0.5000,0.4593,0.4800,0.9374,0.5490,0.3673,0.6923
2,iforest_n200_s1024_f0.5,200,1024,0.5000,0.4471,0.5500,0.9338,0.5278,0.7857,0.4231
3,iforest_n100_s256_f1.0,100,256,1.0000,0.4390,0.3243,0.9158,0.3782,0.5455,0.2308
4,iforest_n100_s1024_f1.0,100,1024,1.0000,0.4375,0.4255,0.9497,0.4336,0.4762,0.3846
5,iforest_n100_s1024_f0.5,100,1024,0.5000,0.4359,0.5000,0.9319,0.5174,0.9000,0.3462
6,iforest_n400_s256_f0.5,400,256,0.5000,0.4356,0.4167,0.9117,0.3970,0.4545,0.3846
7,iforest_n400_s1024_f1.0,400,1024,1.0000,0.4348,0.5455,0.9381,0.4867,0.6667,0.4615
8,iforest_n200_s256_f1.0,200,256,1.0000,0.4222,0.4390,0.9152,0.4481,0.6000,0.3462
9,iforest_n200_s256_f0.5,200,256,0.5000,0.4222,0.4878,0.9109,0.4212,0.6667,0.3846


## 6. View results in MLflow

```
mlflow ui --backend-store-uri file:./mlruns
```

Then open http://127.0.0.1:5000 — every run carries:

- parameters (model name, contamination, scaler, seed),
- val + test metrics (ROC-AUC, PR-AUC, threshold, precision, recall, F1, confusion-matrix counts),
- artifacts: confusion-matrix image, ROC image, fitted model (joblib), dataset card.